In [4]:
REPO_URL = "https://github.com/hcagri/MEGA-GNN"  # change this
!git clone $REPO_URL
%cd MEGA-GNN

Cloning into 'MEGA-GNN'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 90 (delta 21), reused 72 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 2.32 MiB | 5.11 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/MEGA-GNN/MEGA-GNN


In [5]:
# Clean out conflicting packages
!pip -q uninstall -y torch torchvision torchaudio torch-geometric pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv triton nvidia-*

# Fresh, known-good stack for Colab GPU
!pip -q install --upgrade pip setuptools wheel
!pip -q install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip -q install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
!pip -q install torch-geometric
!pip -q install -e genagg

ERROR: Invalid requirement: 'nvidia-*': Expected semicolon (after name with no version specifier) or end
    nvidia-*
          ^
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for genagg (pyproject.toml) ... done


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [8]:
import os

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/math-168/data"
!ls "$DATA_DIR"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
formatted_transactions.csv  HI-Small_Trans.csv	   LI-Small_Trans.csv
HI-Small_accounts.csv	    LI-Small_accounts.csv  Small_HI
HI-Small_Patterns.txt	    LI-Small_Patterns.txt


In [9]:
%cd /content/MEGA-GNN
!pip -q install pybind11

import sysconfig
EXT = sysconfig.get_config_var("EXT_SUFFIX") or ".so"
print("Using extension suffix:", EXT)

!c++ -O3 -Wall -shared -std=c++17 -fPIC $(python -m pybind11 --includes) ports_cpp.cpp -o ports_cpp{EXT}
!c++ -O3 -Wall -shared -std=c++17 -fPIC $(python -m pybind11 --includes) negative_sampling.cpp -o negative_sampling{EXT}

/content/MEGA-GNN
Using extension suffix: .cpython-312-x86_64-linux-gnu.so
negative_sampling.cpp: In function ‘std::vector<std::vector<int> > generate_negative_samples(const std::vector<std::vector<int> >&, const std::vector<std::vector<int> >&, int)’:
negative_sampling.cpp:37:22: warning: comparison of integer expressions of different signedness: ‘int’ and ‘std::vector<int>::size_type’ {aka ‘long unsigned int’} []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wsign-compare-Wsign-compare]8;;]
   37 |     for(int i = 0; i < pos_edge_index[0].size(); i++){
      |                    ~~^~~~~~~~~~~~~~~~~~~~~~~~~~


In [10]:
import ports_cpp, negative_sampling
print("C++ extensions loaded OK")

C++ extensions loaded OK


In [11]:
%cd /content/MEGA-GNN

# Clean broken installs
!pip -q uninstall -y torch torchvision torchaudio pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric

# Install a widely supported combo
!pip -q install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

# Install required PyG extension ops (skip pyg-lib; it's optional)
!pip -q install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
!pip -q install torch-geometric

/content/MEGA-GNN
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
genagg 2.1.1 requires torch_geometric, which is not installed.


In [12]:
import torch, torch_geometric, torch_scatter
print(torch.__version__, torch.version.cuda, torch_geometric.__version__)

2.6.0+cu124 12.4 2.7.0


In [13]:
2# 1) Install genagg from its own project dir
%cd /content/MEGA-GNN/genagg
!pip -q install -e .

# 2) Go back to repo root
%cd /content/MEGA-GNN

# 3) Quick import test using explicit PYTHONPATH to inner package project
import os, sys
sys.path.insert(0, "/content/MEGA-GNN/genagg")
from genagg import GenAgg
print("GenAgg import OK:", GenAgg)

/content/MEGA-GNN/genagg
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for genagg (pyproject.toml) ... done
/content/MEGA-GNN
GenAgg import OK: <class 'genagg.genagg.GenAgg'>


In [14]:
from pathlib import Path

p = Path("/content/MEGA-GNN/models.py")
s = p.read_text()
old = "remapped_edge_attr = torch.index_select(n_edge_attr, 0, inverse_indices) # artificall node attributes"
new = "remapped_edge_attr = n_edge_attr[inverse_indices] # avoid FX/index_select tracing issue"
if old not in s:
    raise RuntimeError("Expected line not found; open models.py and patch manually.")
p.write_text(s.replace(old, new))
print("Patched models.py")

Patched models.py


In [ ]:
2
!PYTHONPATH=/content/MEGA-GNN/genagg:$PYTHONPATH python main.py \
  --data Small_HI \
  --model gin \
  --emlps \
  --reverse_mp \
  --flatten_edges \
  --edge_agg_type gin \
  --task edge_class \
  --n_epochs 10 \
  --batch_size 2048 \
  --num_neighs 20 20 \
  --save_model

2026-03-12 05:13:49,148 [INFO ] Random seed set as 32620
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sophiasharif (sophiasharif-ucla) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ setting up run wa4d54fb (0.2s)
wandb: ⣽ setting up run wa4d54fb (0.2s)
wandb: ⣾ setting up run wa4d54fb (0.2s)
wandb: ⣷ setting up run wa4d54fb (0.2s)
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /content/MEGA-GNN/wandb/run-20260312_051349-wa4d54fb
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Small_HI | gin+EdgeAgg=gin
wandb: ⭐️ View project at https://wandb.ai/sophiasharif-ucla/project_name
wandb: 🚀 View run at https://wandb.ai/sophiasharif-ucla/project_name/runs/wa4d54fb
----- CONFIG -----
epochs : 10
batch_size : 2048
model : gin
data : Small_HI
num_neighbors : [20, 20]
lr : 0.003213266113989207
n_hidden : 6